# Esep safe-6 — public portfolio reproduction

Enable GPU T4 x2 and Internet; use a fresh Kaggle session and Save & Run All. No GitHub token required. This packaging has not had a new GPU validation. Historical results are in the repository evidence folder.


In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/esep-soup-kernels')
assert not repo.exists(), 'Use a fresh Kaggle session to preserve prior outputs.'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/gojosatorukz/esep-soup-kernels.git', str(repo)], check=True)
def run(*args, **kwargs):
    subprocess.run(list(args), cwd=repo, check=True, **kwargs)
run('nvidia-smi')
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt')
run(sys.executable, 'scripts/verify_source.py')


In [ ]:
import os
run(sys.executable, 'task/prepare_data.py')
run(sys.executable, '-m', 'pytest', 'solution', '-q')
run(sys.executable, '-m', 'pytest', 'tests', '-q', env={**os.environ, 'SOUP_APPLY_SOLUTION':'1'})
run(sys.executable, 'scripts/verify_source.py')


In [ ]:
run(sys.executable, 'harness/run_task.py', '--repeats', '2')
run(sys.executable, 'harness/check_rules.py', '--result', 'runs/result.json')
run(sys.executable, 'harness/score.py', 'runs/result.json')
run(sys.executable, 'scripts/verify_source.py')


In [ ]:
import json
result = json.loads((repo/'runs/result.json').read_text())
checks = json.loads((repo/'runs/checks.json').read_text())
assert len(checks['checks']) == 10 and all(row['ok'] for row in checks['checks'])
print(json.dumps(result.get('summary', {}), indent=2))
print('All ten harness checks passed. Download the runs/ directory from this session.')
